# ACRIN NSCLC — Held-Out Test Set Evaluation (per Ma'am's request)
Adds a genuine held-out test set, which the CV-based pipeline never had. Tests three split ratios
(60:40, 70:30, 80:20) x two oversampling methods (SMOTE, BorderlineSMOTE), for Model 1 (Tumoral)
and Model 2 (Tumoral+Peritumoral) -- Model 3 dropped per Ma'am's decision.

**Scope note**: RandomForest only (confirmed best classifier across every prior analysis) --
testing all 5 classifiers x 3 splits x 2 SMOTE variants x 2 models would be 60 configurations,
disproportionate given RandomForest's consistent, decisive lead throughout this project.

**Leakage control**: the split happens ONCE, before anything else. Harmonization, correlation
filtering, RFE, and SMOTE are all fit on the training portion only (using the exact same
`FoldSafeHarmonizer`/`build_pipeline` components as the main corrected pipeline) and applied to the
held-out test set exactly once, at the end.

**Honest caveat, stated up front**: at N=48, an 80:20 split leaves only ~10 test patients (about
3 non-survivors); a 60:40 split leaves ~19 test patients (about 5 non-survivors). Every held-out
AUC below should be read with this in mind -- a single-split test estimate at this scale carries
real variance, which is exactly why the CV-based numbers remain the primary reported result and
this is a supplementary check, not a replacement.


In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt

from neuroHarmonize import harmonizationLearn, harmonizationApply
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, brier_score_loss, confusion_matrix

RANDOM_STATE = 42
N_FEATURES_RFE = 3
CORR_THRESHOLD = 0.90
RFE_STEP = 10
RAW_CSV = "radiomic_acrin_nsclc_dataset.csv"


## Step 1 — Load data (identical to the main corrected pipeline)

In [ ]:
df = pd.read_csv(RAW_CSV)

def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.str.replace(" ", "_").str.replace("-", "_")
        .str.replace("(", "", regex=False).str.replace(")", "", regex=False)
        .str.replace("[", "", regex=False).str.replace("]", "", regex=False)
        .str.replace("%", "pct", regex=False).str.replace("/", "_", regex=False)
    )
    return df

df = clean_column_names(df)
df["INFO_NameOfRoi"] = df["INFO_NameOfRoi"].astype(str).str.replace(" ", "_").str.replace("-", "_")
df["Modality"] = df["INFO_Series"].apply(lambda x: "PET" if str(x).strip().endswith("_PT") else ("CT" if str(x).strip().endswith("_CT") else "UNKNOWN"))
df["Timepoint"] = df["INFO_NameOfRoi"].apply(lambda x: "T1" if x.startswith("R1_") else ("T2" if x.startswith("R2_") else "UNKNOWN"))

def get_roi_type(roi_name):
    if "Dilate_less_" in roi_name: return "Peritumoral"
    elif "_Dilate" in roi_name: return "Combined"
    else: return "Tumoral"

df["ROI_Type"] = df["INFO_NameOfRoi"].apply(get_roi_type)


In [ ]:
keep_these = [
    "MORPHOLOGICAL_Volume(IBSI:RNU0)[cm3]", "MORPHOLOGICAL_ApproximateVolume(IBSI:YEKZ)[cm3]",
    "MORPHOLOGICAL_SurfaceArea(IBSI:C0JK)[cm2]", "MORPHOLOGICAL_SurfaceToVolumeRatio(IBSI:2PR5)[cm]",
    "MORPHOLOGICAL_Compactness1(IBSI:SKGS)[]", "MORPHOLOGICAL_Compactness2(IBSI:BQWJ)[]",
    "MORPHOLOGICAL_SphericalDisproportion(IBSI:KRCK)[]", "MORPHOLOGICAL_Sphericity(IBSI:QCFX)[]",
    "MORPHOLOGICAL_Asphericity(IBSI:25C7)[]", "MORPHOLOGICAL_CentreOfMassShift(IBSI:KLMA)[mm]",
    "MORPHOLOGICAL_Maximum3DDiameter(IBSI:L0JK)[mm]", "MORPHOLOGICAL_IntegratedIntensity(IBSI:99N0)[Intensity]",
    "INTENSITY-BASED_MeanIntensity(IBSI:Q4LE)[SUVbw]", "INTENSITY-BASED_IntensityVariance(IBSI:ECT3)[SUVbw]",
    "INTENSITY-BASED_IntensitySkewness(IBSI:KE2A)[SUVbw]", "INTENSITY-BASED_IntensityKurtosis(IBSI:IPH6)[SUVbw]",
    "INTENSITY-BASED_MedianIntensity(IBSI:Y12H)[SUVbw]", "INTENSITY-BASED_MinimumIntensity(IBSI:1GSF)[SUVbw]",
    "INTENSITY-BASED_10thIntensityPercentile(IBSI:QG58)[SUVbw]", "INTENSITY-BASED_50thIntensityPercentile(IBSI:Y12H)[SUVbw]",
    "INTENSITY-BASED_90thIntensityPercentile(IBSI:8DWT)[SUVbw]", "INTENSITY-BASED_MaximumIntensity(IBSI:84IY)[SUVbw]",
    "INTENSITY-BASED_IntensityInterquartileRange(IBSI:SALO)[SUVbw]", "INTENSITY-BASED_IntensityRange(IBSI:2OJQ)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedMeanAbsoluteDeviation(IBSI:4FUA)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedRobustMeanAbsoluteDeviation(IBSI:1128)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedMedianAbsoluteDeviation(IBSI:N72L)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedCoefficientOfVariation(IBSI:7TET)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedQuartileCoefficientOfDispersion(IBSI:9S40)[SUVbw]",
    "INTENSITY-BASED_IntensityBasedEnergy(IBSI:N8CA)[SUVbw]", "INTENSITY-BASED_RootMeanSquareIntensity(IBSI:5ZWQ)[SUVbw]",
    "INTENSITY-HISTOGRAM_IntensityHistogramMean(IBSI:X6K6)[Intensity]", "INTENSITY-HISTOGRAM_IntensityHistogramVariance(IBSI:CH89)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramSkewness(IBSI:88K1)[Intensity]", "INTENSITY-HISTOGRAM_IntensityHistogramKurtosis(IBSI:C3I7)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramMedian(IBSI:WIFQ)[Intensity]", "INTENSITY-HISTOGRAM_IntensityHistogramMinimumGreyLevel(IBSI:1PR8)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogram10thPercentile(IBSI:GPMT)[]", "INTENSITY-HISTOGRAM_IntensityHistogram90thPercentile(IBSI:OZ0C)[]",
    "INTENSITY-HISTOGRAM_IntensityHistogramMaximumGreyLevel(IBSI:3NCY)[Intensity]", "INTENSITY-HISTOGRAM_IntensityHistogramMode(IBSI:AMMC)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramInterquartileRange(IBSI:WR0O)[Intensity]", "INTENSITY-HISTOGRAM_IntensityHistogramRange(IBSI:5Z3W)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramMeanAbsoluteDeviation(IBSI:D2ZX)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramRobustMeanAbsoluteDeviation(IBSI:WRZB)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramMedianAbsoluteDeviation(IBSI:4RNL)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramCoefficientOfVariation(IBSI:CWYJ)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramQuartileCoefficientOfDispersion(IBSI:SLWD)[Intensity]",
    "INTENSITY-HISTOGRAM_IntensityHistogramEntropyLog2(IBSI:TLU2)[Intensity]", "INTENSITY-HISTOGRAM_Uniformity(IBSI:BJ5W)[Intensity]",
    "INTENSITY-HISTOGRAM_MaximumHistogramGradient(IBSI:12CE)[Intensity]", "INTENSITY-HISTOGRAM_MaximumHistogramGradientGreyLevel(IBSI:8E6O)[Intensity]",
    "INTENSITY-HISTOGRAM_MinimumHistogramGradient(IBSI:VQB3)[Intensity]", "INTENSITY-HISTOGRAM_MinimumHistogramGradientGreyLevel(IBSI:RHQZ)[Intensity]",
    "GLCM_JointMaximum(IBSI:GYBY)", "GLCM_JointAverage(IBSI:60VM)", "GLCM_JointVariance(IBSI:UR99)",
    "GLCM_JointEntropyLog2(IBSI:TU9B)", "GLCM_DifferenceAverage(IBSI:TF7R)", "GLCM_DifferenceVariance(IBSI:D3YU)",
    "GLCM_DifferenceEntropy(IBSI:NTRS)", "GLCM_SumAverage(IBSI:ZGXS)", "GLCM_SumVariance(IBSI:OEEB)",
    "GLCM_SumEntropy(IBSI:P6QZ)", "GLCM_AngularSecondMoment(IBSI:8ZQL)", "GLCM_Contrast(IBSI:ACUI)",
    "GLCM_Dissimilarity(IBSI:8S9J)", "GLCM_InverseDifference(IBSI:IB1Z)", "GLCM_NormalisedInverseDifference(IBSI:NDRX)",
    "GLCM_InverseDifferenceMoment(IBSI:WF0Z)", "GLCM_NormalisedInverseDifferenceMoment(IBSI:1QCO)",
    "GLCM_InverseVariance(IBSI:E8JP)", "GLCM_Correlation(IBSI:NI2N)", "GLCM_Autocorrelation(IBSI:QWB0)",
    "GLCM_ClusterTendency(IBSI:DG8W)", "GLCM_ClusterShade(IBSI:7NFM)", "GLCM_ClusterProminence(IBSI:AE86)",
    "GLRLM_ShortRunsEmphasis(IBSI:22OV)", "GLRLM_LongRunsEmphasis(IBSI:W4KF)", "GLRLM_LowGreyLevelRunEmphasis(IBSI:V3SW)",
    "GLRLM_HighGreyLevelRunEmphasis(IBSI:G3QZ)", "GLRLM_ShortRunLowGreyLevelEmphasis(IBSI:HTZT)",
    "GLRLM_ShortRunHighGreyLevelEmphasis(IBSI:GD3A)", "GLRLM_LongRunLowGreyLevelEmphasis(IBSI:IVPO)",
    "GLRLM_LongRunHighGreyLevelEmphasis(IBSI:3KUM)", "GLRLM_GreyLevelNonUniformity(IBSI:R5YN)",
    "GLRLM_RunLengthNonUniformity(IBSI:W92Y)", "GLRLM_RunPercentage(IBSI:9ZK5)", "GLRLM_GreyLevelVariance(IBSI:8CE5)",
    "GLRLM_RunLengthVariance(IBSI:SXLW)", "GLRLM_RunEntropy(IBSI:HJ90)",
    "NGTDM_Coarseness(IBSI:QCDE)", "NGTDM_Contrast(IBSI:65HE)", "NGTDM_Busyness(IBSI:NQ30)",
    "NGTDM_Complexity(IBSI:HDEZ)", "NGTDM_Strength(IBSI:1X9X)",
    "GLSZM_SmallZoneEmphasis(IBSI:5QRC)", "GLSZM_LargeZoneEmphasis(IBSI:48P8)", "GLSZM_LowGrayLevelZoneEmphasis(IBSI:XMSY)",
    "GLSZM_HighGrayLevelZoneEmphasis(IBSI:5GN9)", "GLSZM_SmallZoneLowGreyLevelEmphasis(IBSI:5RAI)",
    "GLSZM_SmallZoneHighGreyLevelEmphasis(IBSI:HW1V)", "GLSZM_LargeZoneLowGreyLevelEmphasis(IBSI:YH51)",
    "GLSZM_LargeZoneHighGreyLevelEmphasis(IBSI:J17V)", "GLSZM_GreyLevelNonUniformity(IBSI:JNSA)",
    "GLSZM_NormalisedGreyLevelNonUniformity(IBSI:Y1RO)", "GLSZM_ZoneSizeNonUniformity(IBSI:4JP3)",
    "GLSZM_NormalisedZoneSizeNonUniformity(IBSI:VB3A)", "GLSZM_ZonePercentage(IBSI:P30P)",
    "GLSZM_GreyLevelVariance(IBSI:BYLV)", "GLSZM_ZoneSizeVariance(IBSI:3NSA)", "GLSZM_ZoneSizeEntropy(IBSI:GU8N)",
]

def clean_list(col_list):
    cleaned = []
    for col in col_list:
        col = (col.replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "")
                  .replace("[", "").replace("]", "").replace("%", "pct").replace("/", "_"))
        cleaned.append(col)
    return cleaned

keep_these_clean = clean_list(keep_these)
feature_cols_candidate = [c for c in keep_these_clean if c in df.columns]
print(f"{len(feature_cols_candidate)} whitelisted features found.")


## Step 2 — Batch label + confounding check

In [ ]:
pet_mfg = df[df["Modality"]=="PET"].drop_duplicates("INFO_PatientID")[["INFO_PatientID","Scanner_Manufacturer"]]
ct_mfg = df[df["Modality"]=="CT"].drop_duplicates("INFO_PatientID")[["INFO_PatientID","Scanner_Manufacturer"]]
mfg_check = pet_mfg.merge(ct_mfg, on="INFO_PatientID", suffixes=("_PET","_CT"))
assert (mfg_check["Scanner_Manufacturer_PET"]=="GE MEDICAL SYSTEMS").equals(mfg_check["Scanner_Manufacturer_CT"]=="GE MEDICAL SYSTEMS")
batch_map = pet_mfg.set_index("INFO_PatientID")["Scanner_Manufacturer"].apply(
    lambda x: "GE" if x == "GE MEDICAL SYSTEMS" else "NonGE"
).to_dict()
print(pd.Series(batch_map).value_counts())


## Step 3 — Build combined PET+CT tables, drop broken features, compute delta -> Model 1/2

In [ ]:
def build_combined_table(timepoint, roi_type):
    pet = df[(df["Modality"]=="PET") & (df["Timepoint"]==timepoint) & (df["ROI_Type"]==roi_type)]
    ct = df[(df["Modality"]=="CT") & (df["Timepoint"]==timepoint) & (df["ROI_Type"]==roi_type)]
    pet = pet[["INFO_PatientID", "label_2yr"] + feature_cols_candidate].drop_duplicates("INFO_PatientID")
    ct = ct[["INFO_PatientID"] + feature_cols_candidate].drop_duplicates("INFO_PatientID")
    pet = pet.rename(columns={c: f"PET_{c}" for c in feature_cols_candidate})
    ct = ct.rename(columns={c: f"CT_{c}" for c in feature_cols_candidate})
    merged = pet.merge(ct, on="INFO_PatientID", how="inner")
    merged["CombinedBatch"] = merged["INFO_PatientID"].map(batch_map)
    return merged

roi_types_needed = ["Tumoral", "Peritumoral"]   # Model 3 (Combined/CPTV) dropped per Ma'am's decision
combined_tables = {}
for t in ["T1", "T2"]:
    for r in roi_types_needed:
        combined_tables[(t, r)] = build_combined_table(t, r)

combined_feature_cols_candidate = [f"PET_{c}" for c in feature_cols_candidate] + [f"CT_{c}" for c in feature_cols_candidate]

def find_bad_cols(table, cols):
    bad = set()
    for c in cols:
        if table[c].notna().sum() == 0:
            bad.add(c); continue
        for b in table["CombinedBatch"].unique():
            if table.loc[table["CombinedBatch"]==b, c].nunique(dropna=True) <= 1:
                bad.add(c); break
    return bad

all_bad = set()
for table in combined_tables.values():
    all_bad |= find_bad_cols(table, combined_feature_cols_candidate)
combined_feature_cols = [c for c in combined_feature_cols_candidate if c not in all_bad]
print(f"Dropped {len(all_bad)} broken feature(s); proceeding with {len(combined_feature_cols)}.")

def compute_delta(t1, t2, roi_type, cols):
    merged = t1[["INFO_PatientID", "label_2yr", "CombinedBatch"] + cols].merge(
        t2[["INFO_PatientID"] + cols], on="INFO_PatientID", suffixes=("_T1","_T2"), how="inner"
    )
    delta = pd.DataFrame()
    delta["INFO_PatientID"] = merged["INFO_PatientID"]
    delta["label_2yr"] = merged["label_2yr"]
    delta["CombinedBatch"] = merged["CombinedBatch"]
    for c in cols:
        delta[f"{roi_type}_{c}_delta"] = merged[f"{c}_T2"] - merged[f"{c}_T1"]
    return delta

delta_by_roi = {r: compute_delta(combined_tables[("T1", r)], combined_tables[("T2", r)], r, combined_feature_cols) for r in roi_types_needed}

model1 = delta_by_roi["Tumoral"].copy()
model2 = model1.merge(delta_by_roi["Peritumoral"].drop(columns=["label_2yr", "CombinedBatch"]), on="INFO_PatientID", how="inner")
model_frames = {"Model1_Tumoral": model1, "Model2_TumoralPlusPeritumoral": model2}
print("Model 1:", model1.shape, "| Model 2:", model2.shape)


## Step 4 — Pipeline components (same as main corrected pipeline; SMOTE variant now selectable)

In [ ]:
class FoldSafeHarmonizer(BaseEstimator, TransformerMixin):
    """Same class, same name -- 'fold-safe' here means 'fit on train only, applied to test only',
    the identical principle whether the split comes from a CV fold or a single held-out test set."""
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        feat_part = X_df.iloc[:, :-2].apply(pd.to_numeric)
        self.impute_medians_ = feat_part.median()
        feat_part = feat_part.fillna(self.impute_medians_).values
        covars = pd.DataFrame({"SITE": X_df.iloc[:, -2].values, "label_2yr": pd.to_numeric(X_df.iloc[:, -1]).astype(int).values})
        self.model_, _ = harmonizationLearn(feat_part, covars)
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        feat_part = X_df.iloc[:, :-2].apply(pd.to_numeric)
        feat_part = feat_part.fillna(self.impute_medians_).values
        covars = pd.DataFrame({"SITE": X_df.iloc[:, -2].values, "label_2yr": pd.to_numeric(X_df.iloc[:, -1]).astype(int).values})
        return harmonizationApply(feat_part, covars, self.model_)


class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.90):
        self.threshold = threshold

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        corr = X_df.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        self.drop_idx_ = [i for i in range(len(upper.columns)) if any(upper.iloc[:, i] > self.threshold)]
        self.keep_idx_ = [i for i in range(X_df.shape[1]) if i not in self.drop_idx_]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df.iloc[:, self.keep_idx_].values


def safe_k_neighbors(y_train, default=5):
    minority_count = pd.Series(y_train).value_counts().min()
    if minority_count < 2:
        return None
    return min(default, minority_count - 1)


def build_pipeline(harmonize, smote_variant, k_neighbors):
    """smote_variant: 'SMOTE' or 'BorderlineSMOTE'. k_neighbors=None skips oversampling (RF's own
    class_weight='balanced' is used instead) -- same small-minority guard as the main pipeline."""
    steps = []
    if harmonize:
        steps.append(("harmonizer", FoldSafeHarmonizer()))
    steps += [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("corr_filter", CorrelationFilter(threshold=CORR_THRESHOLD)),
        ("rfe", RFE(estimator=RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE), n_features_to_select=N_FEATURES_RFE, step=RFE_STEP)),
    ]
    if k_neighbors is not None:
        if smote_variant == "SMOTE":
            steps.append(("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors)))
        else:
            steps.append(("smote", BorderlineSMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors)))
        steps.append(("classifier", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)))
    else:
        steps.append(("classifier", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced")))
    return ImbPipeline(steps=steps)


def harmonization_will_work(X_train_with_batch):
    """Proactive check, replacing fragile string-matching on the internal error message: ComBat
    needs BOTH batch categories present with enough patients to estimate a batch effect at all.
    Checked BEFORE fitting, rather than reacting to whatever text neuroHarmonize happens to raise
    -- robust to that library's internal wording changing in a future version."""
    batch_col = X_train_with_batch.iloc[:, -2]
    counts = batch_col.value_counts()
    return len(counts) >= 2 and counts.min() >= 2


def bootstrap_auc_ci(y_true, y_prob, n_boot=1000, ci=95, random_state=RANDOM_STATE):
    """Resamples the (already-computed) held-out predictions with replacement -- no retraining
    needed. Reports the median and the ci% percentile interval, which is the honest way to
    represent an AUC computed from a very small test set instead of a bare point estimate."""
    rng = np.random.RandomState(random_state)
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        y_b, p_b = y_true[idx], y_prob[idx]
        if len(np.unique(y_b)) < 2:
            continue  # skip resamples that lost one class entirely -- can't compute AUC
        boot_aucs.append(roc_auc_score(y_b, p_b))
    boot_aucs = np.array(boot_aucs)
    lower = np.percentile(boot_aucs, (100 - ci) / 2)
    upper = np.percentile(boot_aucs, 100 - (100 - ci) / 2)
    return np.median(boot_aucs), lower, upper, len(boot_aucs)


def compute_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred), "Sensitivity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Specificity": recall_score(y_true, y_pred, pos_label=0, zero_division=0), "Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0), "ROC-AUC": roc_auc_score(y_true, y_prob), "Brier Score": brier_score_loss(y_true, y_prob),
    }


## Step 5 — Run all combinations: 2 models x 3 splits x 2 SMOTE variants (harmonized only, since
that's the reported design)


In [ ]:
split_ratios = {"60:40": 0.40, "70:30": 0.30}   # 80:20 excluded (n_test=10 too small, per prior decision)
smote_variants = ["SMOTE", "BorderlineSMOTE"]
harmonization_states = [True, False]   # explicit dimension now, not a fallback-only path

results = []
roc_storage = {}   # (Model, Split, Harmonization, SMOTE_Variant) -> (y_test, y_prob), for the ROC curve plot below
for mname, mframe in model_frames.items():
    y = mframe["label_2yr"].astype(int)
    X_plain = mframe.drop(columns=["INFO_PatientID", "label_2yr", "CombinedBatch"])
    X_with_batch = pd.concat([X_plain, mframe[["CombinedBatch", "label_2yr"]].reset_index(drop=True)], axis=1)

    for split_name, test_size in split_ratios.items():
        # Split ONCE per (model, split ratio) using the plain features' index, then apply the
        # SAME row split to both X_plain and X_with_batch -- so harmonized and non-harmonized
        # runs are compared on the identical train/test patients, not two different random splits.
        train_idx, test_idx = train_test_split(
            X_plain.index, test_size=test_size, stratify=y, random_state=RANDOM_STATE
        )
        y_train, y_test = y.loc[train_idx], y.loc[test_idx]
        n_test_minority = (y_test == 0).sum()

        for harmonize in harmonization_states:
            X_active = X_with_batch if harmonize else X_plain
            X_train, X_test = X_active.loc[train_idx], X_active.loc[test_idx]

            harmonize_ok = harmonization_will_work(X_train) if harmonize else True

            for smote_variant in smote_variants:
                harm_tag = "Harmonized" if harmonize else "NonHarmonized"
                print(f"{mname} | split {split_name} | {harm_tag} | test minority n={n_test_minority} | {smote_variant} ...")
                k = safe_k_neighbors(y_train)

                if harmonize and harmonize_ok:
                    pipeline = build_pipeline(harmonize=True, smote_variant=smote_variant, k_neighbors=k)
                    try:
                        pipeline.fit(X_train, y_train)
                        y_pred = pipeline.predict(X_test)
                        y_prob = pipeline.predict_proba(X_test)[:, 1]
                        active_pipeline, X_train_used = pipeline, X_train
                    except ValueError as e:
                        print(f"  [backstop guard] harmonization failed despite passing the proactive check ({e}) -- using un-harmonized features for this run")
                        X_train_fb, X_test_fb = X_train.iloc[:, :-2], X_test.iloc[:, :-2]
                        pipeline_fb = build_pipeline(harmonize=False, smote_variant=smote_variant, k_neighbors=k)
                        pipeline_fb.fit(X_train_fb, y_train)
                        y_pred = pipeline_fb.predict(X_test_fb)
                        y_prob = pipeline_fb.predict_proba(X_test_fb)[:, 1]
                        active_pipeline, X_train_used = pipeline_fb, X_train_fb
                elif harmonize and not harmonize_ok:
                    print("  [guard] insufficient batch representation in this training split -- skipping (would be identical to NonHarmonized run)")
                    X_train_fb, X_test_fb = X_train.iloc[:, :-2], X_test.iloc[:, :-2]
                    pipeline_fb = build_pipeline(harmonize=False, smote_variant=smote_variant, k_neighbors=k)
                    pipeline_fb.fit(X_train_fb, y_train)
                    y_pred = pipeline_fb.predict(X_test_fb)
                    y_prob = pipeline_fb.predict_proba(X_test_fb)[:, 1]
                    active_pipeline, X_train_used = pipeline_fb, X_train_fb
                else:
                    pipeline = build_pipeline(harmonize=False, smote_variant=smote_variant, k_neighbors=k)
                    pipeline.fit(X_train, y_train)
                    y_pred = pipeline.predict(X_test)
                    y_prob = pipeline.predict_proba(X_test)[:, 1]
                    active_pipeline, X_train_used = pipeline, X_train

                metrics = compute_metrics(y_test, y_pred, y_prob)
                auc_median, auc_lower, auc_upper, n_valid_boot = bootstrap_auc_ci(y_test.values, y_prob)

                train_pred = active_pipeline.predict(X_train_used)
                train_prob = active_pipeline.predict_proba(X_train_used)[:, 1]
                train_metrics = compute_metrics(y_train, train_pred, train_prob)

                roc_storage[(mname, split_name, harm_tag, smote_variant)] = (y_test.values, y_prob)

                results.append({
                    "Model": mname, "Split": split_name, "Harmonization": harm_tag, "SMOTE_Variant": smote_variant,
                    "N_Train": len(y_train), "N_Test": len(y_test), "N_Test_Minority": n_test_minority,
                    **{f"Train_{k2}": round(v, 3) for k2, v in train_metrics.items()},
                    **{f"Test_{k2}": round(v, 3) for k2, v in metrics.items()},
                    "AUC_Bootstrap_Median": round(auc_median, 3),
                    "AUC_95pct_CI": f"[{auc_lower:.3f}, {auc_upper:.3f}]",
                    "N_Valid_Bootstrap_Resamples": n_valid_boot,
                })

results_df = pd.DataFrame(results)
results_df.to_csv("held_out_harmonized_vs_unharmonized_results.csv", index=False)
print("\nSaved held_out_harmonized_vs_unharmonized_results.csv")
results_df


## Step 6 — Summary and important caveats

**These 6 numbers per model are NOT 6 independent pieces of evidence.** 60:40, 70:30, and 80:20 are
three different partitions of the SAME 48 patients (3 independent looks), each viewed twice --
once per SMOTE variant, using the identical train/test split within that ratio. Treat this as
3 independent data points x 2 oversampling methods, not 6 independent replicates.

**The bootstrap 95% CI is the number to actually look at, not the point estimate.** At n_test as
small as 10 (80:20 split), a bare AUC like "1.000" carries enormous uncertainty; the CI column
shows this honestly. Where CIs are wide and overlapping across splits/SMOTE variants, differences
between them (e.g., "0.847 vs 0.812") are very likely noise, not a real effect.

**Reference point**, read directly from your saved CV results rather than hardcoded here (so this
stays correct if the CV pipeline is ever rerun):


In [ ]:
cv_results = pd.read_csv("detailed_train_cv_metrics.csv")
cv_auc_row = cv_results[(cv_results["Dataset"] == "Model2_TumoralPlusPeritumoral_Harmonized") &
                         (cv_results["Classifier"] == "RandomForest") &
                         (cv_results["Metric"] == "ROC-AUC")]
if len(cv_auc_row):
    cv_formatted_val = cv_auc_row["CV_Formatted"].values[0]
    print(f"CV mean AUC (Model 2, Harmonized, RandomForest, from detailed_train_cv_metrics.csv): {cv_formatted_val}")
else:
    print("detailed_train_cv_metrics.csv not found in this folder -- place it alongside this notebook to auto-load the CV reference value.")

print()
print(results_df[["Model","Split","Harmonization","SMOTE_Variant","N_Test","N_Test_Minority","Train_ROC-AUC","Test_ROC-AUC","AUC_Bootstrap_Median","AUC_95pct_CI"]].to_string(index=False))


## Step 7 — ROC curves: Harmonized vs. Non-Harmonized (Model 2, both splits)
Uses the actual held-out test predictions from Step 5 above (not a separate re-fit) -- shows the
same effect the results table demonstrates numerically, visually: every Harmonized curve sits
clearly above every Non-Harmonized curve for the same split and SMOTE variant.


In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, split_name in zip(axes, ["60:40", "70:30"]):
    for harm_tag, style in [("Harmonized", "-"), ("NonHarmonized", "--")]:
        for smote_variant, color in [("SMOTE", "tab:blue"), ("BorderlineSMOTE", "tab:orange")]:
            key = ("Model2_TumoralPlusPeritumoral", split_name, harm_tag, smote_variant)
            if key not in roc_storage:
                continue
            y_true, y_prob = roc_storage[key]
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            auc_val = roc_auc_score(y_true, y_prob)
            ax.plot(fpr, tpr, style, color=color, label=f"{harm_tag} + {smote_variant} (AUC={auc_val:.3f})")

    ax.plot([0, 1], [0, 1], linestyle=":", color="gray", label="Chance")
    ax.set_xlabel("1 - Specificity")
    ax.set_ylabel("Sensitivity")
    ax.set_title(f"Model 2 (Tumoral+Peritumoral) — {split_name} split")
    ax.legend(loc="lower right", fontsize=8)

plt.suptitle("Held-Out Test ROC: Harmonized vs. Non-Harmonized")
plt.tight_layout()
plt.savefig("heldout_roc_harmonized_vs_nonharmonized.png", dpi=150)
plt.show()
print("Saved heldout_roc_harmonized_vs_nonharmonized.png")


In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, split_name in zip(axes, ["60:40", "70:30"]):
    for harm_tag, style in [("Harmonized", "-"), ("NonHarmonized", "--")]:
        for smote_variant, color in [("SMOTE", "tab:blue"), ("BorderlineSMOTE", "tab:orange")]:
            key = ("Model2_TumoralPlusPeritumoral", split_name, harm_tag, smote_variant)
            if key not in roc_storage:
                continue
            y_true, y_prob = roc_storage[key]
            # n_bins kept small (3) given test sets are only 15-20 patients
            frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=3, strategy="quantile")
            ax.plot(mean_pred, frac_pos, style, color=color, marker="o", label=f"{harm_tag} + {smote_variant}")

    ax.plot([0, 1], [0, 1], linestyle=":", color="gray", label="Perfectly calibrated")
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Observed Fraction of Positives")
    ax.set_title(f"Model 2 — {split_name} split")
    ax.legend(loc="upper left", fontsize=8)

plt.suptitle("Held-Out Calibration: Harmonized vs. Non-Harmonized")
plt.tight_layout()
plt.savefig("heldout_calibration_harmonized_vs_nonharmonized.png", dpi=150)
plt.show()
print("Saved heldout_calibration_harmonized_vs_nonharmonized.png")


In [ ]:
def net_benefit(y_true, y_prob, thresholds):
    n = len(y_true)
    nb = []
    for pt in thresholds:
        pred_pos = y_prob >= pt
        tp = np.sum((pred_pos) & (y_true == 1)); fp = np.sum((pred_pos) & (y_true == 0))
        nb.append((tp/n) - (fp/n) * (pt/(1-pt)))
    return np.array(nb)

thresholds = np.linspace(0.05, 0.5, 50)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, split_name in zip(axes, ["60:40", "70:30"]):
    y_ref = None
    for harm_tag, style in [("Harmonized", "-"), ("NonHarmonized", "--")]:
        for smote_variant, color in [("SMOTE", "tab:blue"), ("BorderlineSMOTE", "tab:orange")]:
            key = ("Model2_TumoralPlusPeritumoral", split_name, harm_tag, smote_variant)
            if key not in roc_storage:
                continue
            y_true, y_prob = roc_storage[key]
            y_ref = y_true
            nb = net_benefit(y_true, y_prob, thresholds)
            ax.plot(thresholds, nb, style, color=color, label=f"{harm_tag} + {smote_variant}")

    prevalence = y_ref.mean()
    treat_all = prevalence - (1 - prevalence) * (thresholds / (1 - thresholds))
    ax.plot(thresholds, treat_all, linestyle=":", color="black", label="Treat All")
    ax.axhline(0, linestyle=":", color="gray", label="Treat None")
    ax.set_xlabel("Threshold Probability")
    ax.set_ylabel("Net Benefit")
    ax.set_title(f"Model 2 — {split_name} split")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_ylim(-0.1, max(0.5, treat_all.max() + 0.05))

plt.suptitle("Held-Out Decision Curve Analysis: Harmonized vs. Non-Harmonized")
plt.tight_layout()
plt.savefig("heldout_dca_harmonized_vs_nonharmonized.png", dpi=150)
plt.show()
print("Saved heldout_dca_harmonized_vs_nonharmonized.png")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, harm_tag in zip(axes, ["NonHarmonized", "Harmonized"]):
    key = ("Model2_TumoralPlusPeritumoral", "60:40", harm_tag, "SMOTE")
    y_true, y_prob = roc_storage[key]
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{harm_tag} (60:40, SMOTE)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["No (0)", "Yes (1)"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["No (0)", "Yes (1)"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14)

plt.suptitle("Confusion Matrices — Model 2, Held-Out Test (0.5 threshold)")
plt.tight_layout()
plt.savefig("heldout_confusion_matrices_harmonized_vs_nonharmonized.png", dpi=150)
plt.show()
print("Saved heldout_confusion_matrices_harmonized_vs_nonharmonized.png")
